<a href="https://colab.research.google.com/drive/1oCFDI-rKVox9DPNIz7dR_o1z3EgxNl36?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
file_path = '/content/iproperty_cleaned.csv'

In [2]:
import pandas as pd
import numpy as np
import concurrent.futures
import re
import time
import os
import tracemalloc
import psutil
from IPython.display import display, HTML

In [3]:
df_clean = pd.read_csv(file_path)
display(HTML(f"<b>Dataset Loaded Successfully:</b> {len(df_clean):,} records found."))
display(df_clean.head(3))

,listing_url,property_type,price_rm,price_psf,built_up_sqft,land_area_sqft,furnishing,scraped_at
0,https://www.iproperty.com.my/property/iskandar...,Bungalow/Villa,4900000.0,482.00,7200.0,10166.0,Unfurnished,2026-06-21 20:50:17
1,https://www.iproperty.com.my/property/johor-ba...,Terrace/Link,848000.0,403.81,1600.0,2100.0,Unfurnished,2026-06-21 20:50:17
2,https://www.iproperty.com.my/property/kepong/l...,Terrace/Link,1550000.0,939.39,3600.0,1650.0,Fully Furnished,2026-06-21 20:50:17


## Defining the Architecuture
Since the data is already cleaned of text, our heavy-lifting task will be calculating a new categorical `investment_tier` based on numerical logic.

## **Baseline Model** | Single-Threaded Sequential Core Iteration

In [4]:
def run_baseline_numerical(df):
    df_copy = df.copy()
    investment_categories = []

    for idx, row in df_copy.iterrows():
        price = row['price_rm']
        psf = row['price_psf']

        if pd.isna(price) or pd.isna(psf):
            investment_categories.append("Unknown")
        elif price > 1000000 and psf > 800:
            investment_categories.append("Premium Luxury")
        elif price > 500000 and psf > 400:
            investment_categories.append("Mid-Tier")
        else:
            investment_categories.append("Affordable")

    df_copy['investment_tier'] = investment_categories
    return df_copy

## **Technique 1** | Macro-Parallelism (Process-Based Multiprocessing Loop)

In [5]:
def multi_processing_numerical_kernel(df_shard):
    return run_baseline_numerical(df_shard)

## **Technique 2** | Hybrid Framework (Multiprocessing + Micro-Vectorization)

In [6]:
def hybrid_vectorized_numerical_kernel(df_shard):
    df_copy = df_shard.copy()
    conditions = [
        (df_copy['price_rm'].isna()) | (df_copy['price_psf'].isna()),
        (df_copy['price_rm'] > 1000000) & (df_copy['price_psf'] > 800),
        (df_copy['price_rm'] > 500000) & (df_copy['price_psf'] > 400)
    ]
    choices = ["Unknown", "Premium Luxury", "Mid-Tier"]

    df_copy['investment_tier'] = np.select(conditions, choices, default="Affordable")
    return df_copy

## **Execution & Visual Output**

In [7]:
import tracemalloc
import psutil

def run_performance_averaging_suite(df_data, num_cores, runs=3):
    chunk_size = int(np.ceil(len(df_data) / num_cores))
    df_shards = [df_data.iloc[i * chunk_size : (i + 1) * chunk_size] for i in range(num_cores)]

    architectures = [
        ("Baseline (Sequential)", run_baseline_numerical, False, "before"),
        ("Tech 1 (MP Pool)", multi_processing_numerical_kernel, True, "after"),
        ("Tech 2 (Hybrid Vectorized)", hybrid_vectorized_numerical_kernel, True, "after")
    ]

    before_rows = []
    after_rows = []
    final_processed_df = None
    total_records = len(df_data)

    display(HTML(f"<h3>⏱️ Commencing Benchmarking Suite ({runs} Iterations per Model)...</h3>"))

    for name, target_function, execution_mode, category in architectures:
        print(f"\nEvaluating Model: {name}")
        print("-" * 65)

        run_latencies = []
        run_memories = []
        run_cpus = []

        for run in range(1, runs + 1):
            tracemalloc.start()
            psutil.cpu_percent(interval=None)
            time.sleep(0.2)

            start_latency = time.time()

            if execution_mode:
                with concurrent.futures.ProcessPoolExecutor(max_workers=num_cores) as executor:
                    shards_out = list(executor.map(target_function, df_shards))
                    if name == "Tech 2 (Hybrid Vectorized)" and final_processed_df is None:
                        final_processed_df = pd.concat(shards_out, ignore_index=True)
            else:
                res = target_function(df_data)
                if final_processed_df is None:
                    final_processed_df = res

            elapsed_latency = time.time() - start_latency
            cpu_utilization = psutil.cpu_percent(interval=None)
            _, peak_memory_bytes = tracemalloc.get_traced_memory()
            tracemalloc.stop()

            peak_memory_mb = peak_memory_bytes / (1024 * 1024)
            run_throughput = total_records / elapsed_latency

            run_latencies.append(elapsed_latency)
            run_memories.append(peak_memory_mb)
            run_cpus.append(cpu_utilization)

            raw_record = {
                "Architecture": f"{name} (Run {run})",
                "Latency (s)": round(elapsed_latency, 4),
                "Memory (MB)": round(peak_memory_mb, 2),
                "CPU Load (%)": round(cpu_utilization, 1),
                "Throughput (records/s)": round(run_throughput, 2)
            }

            if category == "before":
                before_rows.append(raw_record)
            else:
                after_rows.append(raw_record)

            print(f" -> Run {run}/{runs}: Latency = {elapsed_latency:.3f}s | Peak Mem = {peak_memory_mb:.2f}MB | CPU = {cpu_utilization:.1f}% | Throughput = {run_throughput:.1f} rec/s")

        # Compute averages across all iterations
        avg_latency = np.mean(run_latencies)
        avg_memory = np.mean(run_memories)
        avg_cpu = np.mean(run_cpus)
        avg_throughput = total_records / avg_latency

        avg_record = {
            "Architecture": f"{name} (AVERAGE)",
            "Latency (s)": round(avg_latency, 4),
            "Memory (MB)": round(avg_memory, 2),
            "CPU Load (%)": round(avg_cpu, 1),
            "Throughput (records/s)": round(avg_throughput, 2)
        }

        if category == "before":
            before_rows.append(avg_record)
        else:
            after_rows.append(avg_record)

    return pd.DataFrame(before_rows), pd.DataFrame(after_rows), final_processed_df

computed_cores = os.cpu_count() or 4
df_before, df_after, df_final_output = run_performance_averaging_suite(df_clean, computed_cores, runs=3)

df_before.to_csv('performance_before.csv', index=False)
df_after.to_csv('performance_after.csv', index=False)

display(HTML("<br><h3>✅ Execution Finished. Metrics completely saved:</h3>"))
print("\n[ performance_before.csv (Baseline Data) ]")
display(df_before)
print("\n[ performance_after.csv (Optimized Data) ]")
display(df_after)

display(HTML("<hr><h3>📊 Total Count Breakdown per Investment Tier</h3>"))
df_distribution = pd.DataFrame(df_final_output['investment_tier'].value_counts()).rename(columns={'count': 'Number of Properties'})
display(df_distribution)


Evaluating Model: Baseline (Sequential)
-----------------------------------------------------------------
 -> Run 1/3: Latency = 16.971s | Peak Mem = 26.32MB | CPU = 72.0% | Throughput = 5598.1 rec/s
 -> Run 2/3: Latency = 13.893s | Peak Mem = 26.32MB | CPU = 59.7% | Throughput = 6838.0 rec/s
 -> Run 3/3: Latency = 13.523s | Peak Mem = 26.32MB | CPU = 59.5% | Throughput = 7025.1 rec/s

Evaluating Model: Tech 1 (MP Pool)
-----------------------------------------------------------------
 -> Run 1/3: Latency = 14.786s | Peak Mem = 34.01MB | CPU = 97.9% | Throughput = 6425.3 rec/s
 -> Run 2/3: Latency = 14.643s | Peak Mem = 33.52MB | CPU = 97.9% | Throughput = 6488.2 rec/s
 -> Run 3/3: Latency = 13.872s | Peak Mem = 33.52MB | CPU = 98.1% | Throughput = 6848.7 rec/s

Evaluating Model: Tech 2 (Hybrid Vectorized)
-----------------------------------------------------------------
 -> Run 1/3: Latency = 0.714s | Peak Mem = 38.62MB | CPU = 65.9% | Throughput = 133020.0 rec/s
 -> Run 2/3: Latency


[ performance_before.csv (Baseline Data) ]


,Architecture,Latency (s),Memory (MB),CPU Load (%),Throughput (records/s)
0,Baseline (Sequential) (Run 1),16.9709,26.32,72.0,5598.06
1,Baseline (Sequential) (Run 2),13.8935,26.32,59.7,6838.02
2,Baseline (Sequential) (Run 3),13.5234,26.32,59.5,7025.14
3,Baseline (Sequential) (AVERAGE),14.7959,26.32,63.7,6420.95



[ performance_after.csv (Optimized Data) ]


,Architecture,Latency (s),Memory (MB),CPU Load (%),Throughput (records/s)
0,Tech 1 (MP Pool) (Run 1),14.7859,34.01,97.9,6425.33
1,Tech 1 (MP Pool) (Run 2),14.6427,33.52,97.9,6488.16
2,Tech 1 (MP Pool) (Run 3),13.8718,33.52,98.1,6848.73
3,Tech 1 (MP Pool) (AVERAGE),14.4334,33.68,98.0,6582.22
4,Tech 2 (Hybrid Vectorized) (Run 1),0.7142,38.62,65.9,133020.05
5,Tech 2 (Hybrid Vectorized) (Run 2),0.7721,38.62,69.7,123044.76
6,Tech 2 (Hybrid Vectorized) (Run 3),1.3470,38.62,94.8,70528.35
7,Tech 2 (Hybrid Vectorized) (AVERAGE),0.9445,38.62,76.8,100591.87


,Number of Properties
investment_tier,
Mid-Tier,42271
Affordable,36974
Premium Luxury,15759
